# Spending Effectiveness: Instruction vs. Administration

**Research questions:**
- Where does additional spending appear most effective?
- Does instructional spending outperform administrative spending as a predictor of outcomes?

Total per-pupil spending is only weakly correlated with outcomes once demographics are controlled. But *how* money is allocated — to teachers and instruction vs. administration and overhead — may matter more than the total amount.


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Add project source to path so pipeline/reports can be imported
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

PANEL_PATH = REPO / "data" / "processed" / "district_year_panel.csv"


In [ ]:
def make_synthetic_panel(n_districts: int = 300, seed: int = 42) -> pd.DataFrame:
    """Generate a realistic synthetic district-year panel for exploration."""
    rng = np.random.default_rng(seed)
    years = list(range(2015, 2023))
    states = ["AL", "MA", "TX", "CA", "OH", "NY", "WA", "GA", "FL", "IL"]
    urbanicity = rng.choice(["City", "Suburb", "Town", "Rural"], n_districts, p=[0.20, 0.30, 0.25, 0.25])
    state = [states[i % len(states)] for i in range(n_districts)]

    poverty0    = rng.beta(2, 7, n_districts)
    income0     = np.clip(90_000 - 220_000 * poverty0 + rng.normal(0, 8_000, n_districts), 28_000, 130_000)
    ba0         = np.clip(0.38 - 0.60 * poverty0 + rng.normal(0, 0.05, n_districts), 0.08, 0.75)
    spending0   = np.clip(7_000 + (income0 / 100_000) * 7_000 + rng.normal(0, 2_500, n_districts), 5_000, 32_000)
    instr_share = rng.uniform(0.52, 0.70, n_districts)
    admin_share = rng.uniform(0.05, 0.13, n_districts)
    overperform = rng.normal(0, 0.07, n_districts)   # persistent district "true quality" residual

    rows = []
    capital_spike_years: dict[str, int] = {}
    for di in range(n_districts):
        spike_yr = None
        if rng.random() < 0.18:           # ~18% of districts get a capital investment spike
            spike_yr = years[int(rng.integers(2, len(years) - 3))]
            capital_spike_years[f"D{di:04d}"] = spike_yr

        for yi, yr in enumerate(years):
            poverty  = float(np.clip(poverty0[di] + rng.normal(0, 0.008), 0.02, 0.55))
            income   = float(np.clip(income0[di] + 800 * yi + rng.normal(0, 800), 25_000, 140_000))
            ba       = float(np.clip(ba0[di] + 0.002 * yi + rng.normal(0, 0.004), 0.08, 0.75))
            spending = float(np.clip(spending0[di] + 250 * yi + rng.normal(0, 400), 4_500, 35_000))
            instr_pp = spending * instr_share[di]
            admin_pp = spending * admin_share[di]

            capital_pp = spending * float(rng.beta(1, 10)) * 0.12
            if spike_yr and yr == spike_yr:
                capital_pp = spending * float(rng.uniform(0.18, 0.35))

            # Outcome composite: demographics + instruction share + lagged capital + noise
            composite = (
                0.72
                - 1.10 * poverty
                + 0.25 * ba
                + 0.04 * (income / 80_000 - 1.0)
                + overperform[di]
                + 0.12 * (instr_share[di] - 0.61)
                + (0.045 if (spike_yr and yr >= spike_yr + 3) else 0.0)
                + 0.001 * yi
                + float(rng.normal(0, 0.025))
            )
            composite = float(np.clip(composite, 0.12, 0.97))

            rows.append({
                "district_id":           f"D{di:04d}",
                "district_name":         f"Demo District {di + 1}",
                "year":                  yr,
                "state":                 state[di],
                "urbanicity":            urbanicity[di],
                "enrollment":            int(np.clip(rng.lognormal(7.8, 1.0), 200, 100_000)),
                "poverty_rate":          poverty,
                "median_income":         income,
                "adult_ba_plus_rate":    ba,
                "spending_per_student":  spending,
                "instruction_spending_pp": instr_pp,
                "admin_spending_pp":     admin_pp,
                "capital_outlay_pp":     float(capital_pp),
                "instruction_share":     float(instr_share[di]),
                "admin_share":           float(admin_share[di]),
                "overperform_true":      float(overperform[di]),
                "math_proficiency_rate":    float(np.clip(composite * 0.92 + rng.normal(0, 0.02), 0.10, 0.98)),
                "reading_proficiency_rate": float(np.clip(composite * 0.94 + rng.normal(0, 0.02), 0.10, 0.98)),
                "graduation_rate":          float(np.clip(0.65 + composite * 0.35 + rng.normal(0, 0.015), 0.45, 0.99)),
                "attendance_rate":          float(np.clip(0.82 + composite * 0.18 + rng.normal(0, 0.010), 0.70, 0.99)),
                "capital_spike_year":    spike_yr,
            })

    df = pd.DataFrame(rows)
    df._capital_spike_years = capital_spike_years  # stash for infrastructure notebook
    return df


def load_panel() -> tuple[pd.DataFrame, bool]:
    """Load the real panel if it exists, otherwise generate synthetic data."""
    if PANEL_PATH.exists():
        df = pd.read_csv(PANEL_PATH, dtype={"district_id": str, "county_fips": str})
        print(f"Loaded real panel: {len(df):,} rows from {PANEL_PATH}")
        return df, True
    else:
        df = make_synthetic_panel()
        print(
            f"Real panel not found at {PANEL_PATH}.\n"
            f"Using synthetic data ({len(df):,} rows) — run eol-build-panel to use real data."
        )
        return df, False


def outcome_composite(df: pd.DataFrame) -> pd.Series:
    """Weighted mean of available outcome metrics (mirrors reports.py weights)."""
    weights = {
        "math_proficiency_rate":    1.0,
        "reading_proficiency_rate": 1.0,
        "graduation_rate":          1.5,
        "attendance_rate":          0.5,
    }
    num = pd.Series(0.0, index=df.index)
    den = pd.Series(0.0, index=df.index)
    for col, w in weights.items():
        if col in df.columns:
            valid = pd.to_numeric(df[col], errors="coerce")
            mask  = valid.notna()
            num  += valid.where(mask, 0) * w
            den  += mask.astype(float) * w
    return (num / den.replace(0, np.nan)).rename("outcome_composite")


In [ ]:
df, is_real = load_panel()
df["outcome_composite"] = outcome_composite(df)

# Use the latest year for cross-sectional analysis; multi-year for trends
latest_year = df["year"].max()
cs = df[df["year"] == latest_year].copy()
cs = cs.dropna(subset=["outcome_composite", "spending_per_student",
                        "instruction_spending_pp", "admin_spending_pp"])

cs["instruction_share"] = cs["instruction_spending_pp"] / cs["spending_per_student"]
cs["admin_share"]       = cs["admin_spending_pp"]       / cs["spending_per_student"]

print(f"Year: {latest_year}  |  Districts with spending data: {len(cs):,}")
print(f"\nMedian spending per student: ${cs['spending_per_student'].median():,.0f}")
print(f"Median instruction share:    {cs['instruction_share'].median():.1%}")
print(f"Median admin share:          {cs['admin_share'].median():.1%}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def poverty_color(s):
    return plt.cm.RdYlGn_r(s / s.max())

# Panel A: Total spending vs. outcomes
ax = axes[0]
sc = ax.scatter(
    cs["spending_per_student"] / 1000, cs["outcome_composite"],
    c=cs["poverty_rate"], cmap="RdYlGn_r", alpha=0.5, s=25, vmin=0.05, vmax=0.45
)
plt.colorbar(sc, ax=ax, label="Poverty Rate", shrink=0.8)
m, b = np.polyfit(cs["spending_per_student"] / 1000, cs["outcome_composite"], 1)
x_r = np.linspace(cs["spending_per_student"].min() / 1000, cs["spending_per_student"].max() / 1000, 100)
ax.plot(x_r, m * x_r + b, "k--", linewidth=1.5, label=f"slope={m:.3f}")
ax.set_xlabel("Spending per Student ($k)")
ax.set_ylabel("Outcome Composite")
ax.set_title("Total Spending vs. Outcomes\n(Weak raw correlation)")
ax.legend(fontsize=9)

# Panel B: Instruction share vs. outcomes
ax = axes[1]
ax.scatter(
    cs["instruction_share"] * 100, cs["outcome_composite"],
    c=cs["poverty_rate"], cmap="RdYlGn_r", alpha=0.5, s=25, vmin=0.05, vmax=0.45
)
m2, b2 = np.polyfit(cs["instruction_share"] * 100, cs["outcome_composite"], 1)
x_r2 = np.linspace(cs["instruction_share"].min() * 100, cs["instruction_share"].max() * 100, 100)
ax.plot(x_r2, m2 * x_r2 + b2, "k--", linewidth=1.5, label=f"slope={m2:.4f}")
ax.set_xlabel("Instruction Spending Share (%)")
ax.set_ylabel("Outcome Composite")
ax.set_title("Instruction Share vs. Outcomes\n(Positive relationship)")
ax.legend(fontsize=9)

# Panel C: Admin share vs. outcomes
ax = axes[2]
ax.scatter(
    cs["admin_share"] * 100, cs["outcome_composite"],
    c=cs["poverty_rate"], cmap="RdYlGn_r", alpha=0.5, s=25, vmin=0.05, vmax=0.45
)
m3, b3 = np.polyfit(cs["admin_share"] * 100, cs["outcome_composite"], 1)
x_r3 = np.linspace(cs["admin_share"].min() * 100, cs["admin_share"].max() * 100, 100)
ax.plot(x_r3, m3 * x_r3 + b3, "k--", linewidth=1.5, label=f"slope={m3:.4f}")
ax.set_xlabel("Admin Spending Share (%)")
ax.set_ylabel("Outcome Composite")
ax.set_title("Admin Share vs. Outcomes\n(Near-zero or negative)")
ax.legend(fontsize=9)

plt.suptitle(f"Spending Composition and Outcomes  |  Year {latest_year}", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## Partial correlations — controlling for poverty

Raw correlations conflate the spending-outcome relationship with demographic sorting (high-income districts both spend more and have better outcomes for unrelated reasons). Partial correlations isolate the spending signal after removing poverty's influence.


In [ ]:
def partial_corr(x: pd.Series, y: pd.Series, controls: pd.DataFrame) -> float:
    """Pearson correlation of x and y after linearly residualizing on controls."""
    ctrl = controls.values
    X_ctrl = np.column_stack([np.ones(len(ctrl)), ctrl])
    def resid(v):
        b, *_ = np.linalg.lstsq(X_ctrl, v.values, rcond=None)
        return v.values - X_ctrl @ b
    rx, ry = resid(x), resid(y)
    return float(np.corrcoef(rx, ry)[0, 1])

ctrl = cs[["poverty_rate", "median_income"]]

metrics = {
    "Total spending ($k)":        cs["spending_per_student"] / 1000,
    "Instruction spending ($k)":  cs["instruction_spending_pp"] / 1000,
    "Admin spending ($k)":        cs["admin_spending_pp"] / 1000,
    "Instruction share (%)":      cs["instruction_share"] * 100,
    "Admin share (%)":            cs["admin_share"] * 100,
}

pcorrs = {label: partial_corr(series, cs["outcome_composite"], ctrl) for label, series in metrics.items()}

fig, ax = plt.subplots(figsize=(9, 4))
labels = list(pcorrs.keys())
values = list(pcorrs.values())
colors = ["#2a9d8f" if v > 0 else "#e76f51" for v in values]
bars = ax.barh(labels, values, color=colors, alpha=0.8, edgecolor="white")
ax.axvline(0, color="black", linewidth=1)
for bar, val in zip(bars, values):
    ax.text(val + (0.003 if val >= 0 else -0.003), bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center", ha="left" if val >= 0 else "right", fontsize=10)
ax.set_xlabel("Partial Correlation with Outcome Composite\n(controlling for poverty & income)")
ax.set_title("Which Spending Dimensions Predict Better Outcomes?\n(After controlling for demographics)", fontweight="bold")
ax.set_xlim(-0.25, 0.45)
plt.tight_layout()
plt.show()


## Spending decomposition by urbanicity

In [ ]:
urb_order = ["City", "Suburb", "Town", "Rural"]
urb_present = [u for u in urb_order if u in cs["urbanicity"].unique()]

cols = ["instruction_spending_pp", "admin_spending_pp", "capital_outlay_pp"]
col_labels = ["Instruction", "Administration", "Capital Outlay"]

by_urb = cs.groupby("urbanicity")[cols].median().loc[urb_present] / 1000  # in $k

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(urb_present))
width = 0.25
palette = ["#264653", "#2a9d8f", "#e9c46a"]

for i, (col, label, color) in enumerate(zip(cols, col_labels, palette)):
    ax.bar(x + i * width, by_urb[col], width, label=label, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(urb_present)
ax.set_ylabel("Median Per-Pupil Spending ($k)")
ax.set_title("Per-Pupil Spending Decomposition by Urbanicity", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## Efficiency frontier — high outcomes at lower spending

In [ ]:
# Poverty-adjusted residual: outcomes above what poverty alone predicts
cs["poverty_adj_outcome"] = cs["outcome_composite"] - np.polyval(
    np.polyfit(cs["poverty_rate"], cs["outcome_composite"], 1), cs["poverty_rate"]
)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    cs["spending_per_student"] / 1000,
    cs["poverty_adj_outcome"],
    c=cs["instruction_share"],
    cmap="YlGn",
    alpha=0.6,
    s=35,
    vmin=0.50,
    vmax=0.72,
)
plt.colorbar(sc, ax=ax, label="Instruction Spending Share")
ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.4)
ax.axvline(cs["spending_per_student"].median() / 1000, color="grey", linewidth=1, linestyle=":", alpha=0.6)
ax.set_xlabel("Spending per Student ($k)")
ax.set_ylabel("Poverty-Adjusted Outcome Residual")
ax.set_title(
    "Efficiency Frontier: Poverty-Adjusted Outcomes vs. Total Spending\n"
    "Color = Instruction Share  |  Upper-left quadrant = efficient districts",
    fontweight="bold"
)

# Annotate efficient quadrant
ax.fill_betweenx(
    [cs["poverty_adj_outcome"].quantile(0.70), cs["poverty_adj_outcome"].max() * 1.05],
    ax.get_xlim()[0], cs["spending_per_student"].median() / 1000,
    alpha=0.05, color="#2a9d8f"
)
ax.text(
    cs["spending_per_student"].quantile(0.15) / 1000, cs["poverty_adj_outcome"].quantile(0.85),
    "Efficient\nhigh-instruction\ndistricts", ha="center", va="center",
    color="#1a6b60", fontsize=10, fontweight="bold", alpha=0.8
)
plt.tight_layout()
plt.show()


## Takeaways

- **Total spending has a weak raw correlation with outcomes** — demographics explain most of it. Once poverty is controlled, the relationship is even weaker.
- **Instruction share has a positive partial correlation** with outcomes, even after controlling for demographics. Districts that put more dollars in front of students (vs. overhead) tend to do better.
- **Admin share has a near-zero or slightly negative partial correlation** — this doesn't mean administration is useless, but marginal administrative spending shows little outcome return in the data.
- **The efficient frontier** (upper-left quadrant of the scatter) is dominated by high-instruction-share districts — they achieve above-average outcomes at or below average spending.
- **Practical implication:** Where to look first when a district needs to improve outcomes without a budget increase: is the instruction share below the peer median?
